# Build the network of a region

scgraph publishes networks of the whole world, ready to load (see the notebook *Routes on a multimodal network*). Their data is that of the day of their build.

To get the data of today for one region, build its network on your machine. One call does it: the pipeline `osm_region_multimodal` reads the newest OpenStreetMap extract of the region. The build is also the place to start from when you want rules of your own, for example other kinds of roads.

This notebook builds the network of Denmark, then loads the streets of a part of Paris:

1. Build the network with one call.
2. Read what the network file says about itself.
3. Compare it with the released network of Europe.
4. Compute routes on some node modes.
5. Check the network against known routes.
6. Load the streets of a neighbourhood, with their turn restrictions.

The build downloads about 520 MB, mostly the OpenStreetMap extract of Denmark. It takes about one minute after the downloads, and about 2 GB of memory.

GitHub does not show the maps of a notebook. To see them, run the notebook, or open it in [nbviewer](https://nbviewer.org/github/connor-makowski/scgraph/blob/main/examples/build_a_region.ipynb).

## Setup

The build reads OpenStreetMap files with `osmium`, which the extra `osm` of scgraph installs. The maps need [folium](https://python-visualization.github.io/folium/).

In [1]:
# Until scgraph 4.0 is on PyPI, install it from the `dev` branch:
# %pip install "scgraph[osm] @ git+https://github.com/connor-makowski/scgraph@dev" folium
import logging
import os

import folium

from scgraph import GeoGraph
from scgraph.pipelines import osm_region_multimodal
from scgraph.pipelines.steps.step4_validation_cases import load_validation_cases, validate_network

## 1. Build the network

`build_geograph` takes the name of a pipeline and a region. The region is the path of an extract on [Geofabrik](https://download.geofabrik.de), a server of OpenStreetMap extracts: `europe/denmark`, `europe/france/bretagne`, `north-america/us/texas`.

The first call downloads the sources and builds the network. It keeps the network in the cache of scgraph. scgraph reports its work through `logging`: the next cell shows these messages.

In [2]:
logging.basicConfig(format="%(message)s")
logging.getLogger("scgraph.pipelines").setLevel(logging.INFO)

denmark = GeoGraph.build_geograph("osm_region_multimodal", region="europe/denmark")

logging.getLogger("scgraph.pipelines").setLevel(logging.WARNING)  # the cells below stay quiet

A later call loads the network from the cache. It makes no download and builds nothing:

In [3]:
denmark = GeoGraph.build_geograph("osm_region_multimodal", region="europe/denmark")
print(f"{len(denmark.nodes):,} nodes")

51,719 nodes


## 2. What the network file says about itself

The build wrote one network file. Its header records the pipeline, the sources with the date of their data, and the credit to cite. `scgraph_file` gives the path of the file, and `get_scgraph_info` reads its header.

The header also lists the **metrics** of the file. A metric is one weight for each edge, under a name. The first metric is `length`, with its unit.

In [4]:
file = denmark.scgraph_file
info = GeoGraph.get_scgraph_info(file)

print(f"File: {os.path.basename(file)[:16]}....scgraph, {os.path.getsize(file) / 1e6:.1f} MB")
print(f"{info['nodes']:,} nodes and {info['edges']:,} edges. The lengths are in {info['metrics'][0]['unit']}.")
print("Pipeline:", info["pipeline"])
print("Newest source data:", info["data_up_to"])
print()
for source in info["sources"]:
    print(f"    {source['name']:<15}{source.get('version') or '':<45.44}{source['credit'][:40]}")
print()
print("Credit to cite with your results:", info["attribution"])

File: e82c23f6b60032f6....scgraph, 4.2 MB
51,719 nodes and 173,849 edges. The lengths are in km.
Pipeline: {'name': 'osm_region_multimodal', 'region': 'europe/denmark', 'version': 7}
Newest source data: 2026-11-24T21:21:23Z

    sea                                                         Eurostat, SeaRoute maritime network (fro
    land           5.1.2                                        Natural Earth (1:10m)
    lakes          5.1.2                                        Natural Earth (1:10m)
    rivers         5.1.2                                        Natural Earth (1:10m)
    ports          UpdatedPub150.csv, SHA-256 315644f1e7796629  National Geospatial-Intelligence Agency,
    airports       airports.csv at commit 1cdbf9cbed1e47bed8f11 OurAirports
    flights        flights of 2021: OpenSky v21.12; T-100 and E The OpenSky Network, flight list (Strohm
    rail_terminals query of 2026-11-16                          Wikidata
    osm            2026-11-24T21:21:23Z              

The name of the file is the SHA-256 digest of its content. The Geofabrik extract changes every day, so a build on another day gives another network. Record this digest, or keep the file, when a result must be found again later.

Each node has a **node mode**: the network that it belongs to, or the kind of hub that it is. `get_nodes(node_mode=...)` lists the nodes of one node mode.

In [5]:
for node_mode in denmark.node_modes:
    print(f"{node_mode['name']:<14}{len(denmark.get_nodes(node_mode=node_mode['name'])):>7,} nodes")

road            7,561 nodes
rail            1,927 nodes
sea            42,157 nodes
port               63 nodes
airport            10 nodes
rail_terminal       1 nodes


The roads and the railways are those of Denmark. The sea lanes are those of the whole world, so a route can leave Denmark by sea. A port, an airport or a rail terminal is a hub: a place where a route changes network.

## 3. Compare with the released network

The released network `europe` covers Denmark too. The two networks hold the same kinds of roads. They differ in two ways:

- **The date of the data.** A released network has the data of its release. The network built here has the data of the newest extract.
- **The nodes along a road.** Both networks have a node at each junction. Along a road without a junction, the released network has a node at most about every 20 km. The network built here has one at most about every 4 km.

The next cell reads the date of the data of the two networks, and counts their road nodes in the same rectangle around Denmark. `get_nodes(bbox=...)` lists the nodes inside a rectangle `(south, west, north, east)`.

In [6]:
europe = GeoGraph.load_geograph("europe", node_modes=["road"], load_intermediate_nodes=False)
denmark_roads = GeoGraph.build_geograph(
    "osm_region_multimodal", region="europe/denmark", node_modes=["road"]
)

DENMARK = (54.5, 8.0, 57.8, 12.8)  # south, west, north, east: Denmark without the island of Bornholm

released_nodes = europe.get_nodes(bbox=DENMARK)
built_nodes = denmark_roads.get_nodes(bbox=DENMARK)
released_info = GeoGraph.get_scgraph_info(europe.scgraph_file)
print(f"Released network `europe`: data up to {released_info['data_up_to']}, {len(released_nodes):,} road nodes in the rectangle")
print(f"Network built here:        data up to {info['data_up_to']}, {len(built_nodes):,} road nodes in the rectangle")

Released network `europe`: data up to 2026-11-02T00:59:53Z, 7,503 road nodes in the rectangle
Network built here:        data up to 2026-11-24T21:21:23Z, 7,525 road nodes in the rectangle


The map shows the road nodes of the two networks around Aarhus. A large black circle is a node of the released network. A small orange dot is a node of the network built here.

In [7]:
def make_map(south, west, north, east, height=440):
    """A new map that shows the area between two latitudes and two longitudes."""
    new_map = folium.Map(tiles=None)
    # `control=False` keeps the background out of the control that shows or hides layers
    folium.TileLayer("OpenStreetMap", control=False).add_to(new_map)
    new_map.fit_bounds([[south, west], [north, east]])
    # The figure gives the map its height in the notebook
    folium.Figure(height=height).add_child(new_map)
    return new_map


AARHUS_AREA = (56.0, 9.9, 56.3, 10.5)

node_map = make_map(*AARHUS_AREA)
for node in denmark_roads.get_nodes(bbox=AARHUS_AREA):
    point = [node["latitude"], node["longitude"]]
    folium.CircleMarker(point, radius=2, color="#eb6834", fill_opacity=1).add_to(node_map)
for node in europe.get_nodes(bbox=AARHUS_AREA):
    point = [node["latitude"], node["longitude"]]
    folium.CircleMarker(point, radius=8, color="#0b0b0b", weight=2, fill=False).add_to(node_map)
node_map

The two networks have the same junctions. The network built here has more nodes along the roads between two junctions. In a country with many junctions, such as Denmark, the two counts are close.

## 4. Routes on some node modes

A route is the shortest in kilometres. It starts and ends on a road: scgraph joins each place to its closest road node. With all the node modes, the route can then take a railway, a sea lane or a flight. `node_modes` loads some node modes of the file only: here the road nodes and the edges between them, which include the vehicle ferries.

The example goes from Aalborg, in the north of Denmark, to Copenhagen. `output_legs=True` adds the **legs** of the route to the result. Each edge has an **edge mode**: `road`, `rail`, `sea` or `air`. A leg is a part of the route on one edge mode. The first leg and the last leg have the mode `off_graph`: they are the straight lines between the two places and the network.

In [8]:
AALBORG = {"latitude": 57.0488, "longitude": 9.9217}
COPENHAGEN = {"latitude": 55.6761, "longitude": 12.5683}

routes = {}
for title, geograph in [("All node modes", denmark), ('node_modes=["road"]', denmark_roads)]:
    route = geograph.get_shortest_path(
        origin_node=AALBORG, destination_node=COPENHAGEN, output_legs=True
    )
    routes[title] = route
    legs = ", ".join(f"{leg['edge_mode']} {leg['length']:,.1f} km" for leg in route["legs"])
    print(f"{title}: {route['length']:,.1f} km")
    print(f"    {legs}")

All node modes: 262.9 km
    off_graph 1.0 km, road 1.3 km, sea 253.2 km, road 6.5 km, off_graph 0.8 km
node_modes=["road"]: 321.6 km
    off_graph 1.0 km, road 319.8 km, off_graph 0.8 km


In [9]:
COLORS = {"All node modes": "#0b0b0b", 'node_modes=["road"]': "#eb6834"}

route_map = make_map(55.3, 9.0, 57.3, 13.0, height=520)
for title, route in routes.items():
    layer = folium.FeatureGroup(name=f"{title}: {route['length']:,.1f} km")
    folium.PolyLine(
        route["coordinate_path"],
        color=COLORS[title],
        weight=4,
        tooltip=f"{title}: {route['length']:,.1f} km",
    ).add_to(layer)
    layer.add_to(route_map)
for name, place in [("Aalborg", AALBORG), ("Copenhagen", COPENHAGEN)]:
    folium.CircleMarker(
        [place["latitude"], place["longitude"]],
        radius=4,
        color="#0b0b0b",
        fill_opacity=1,
        tooltip=folium.Tooltip(name, permanent=True, direction="top"),
    ).add_to(route_map)
folium.LayerControl(collapsed=False).add_to(route_map)
route_map

With all the node modes, the shortest route goes by road to a port of Aalborg, then by sea to Copenhagen, then by road again. No traveller goes this way, but it is the shortest line in kilometres. By road, the route crosses the sea from Aarhus on a vehicle ferry, which has the edge mode `road`.

Choose the node modes that fit your question. A metric finds the fastest route in place of the shortest one: the notebook *Routes on a multimodal network* shows how.

## 5. Check the network against known routes

A pipeline comes with **validation cases**. A case describes one fact of the real world that the network must reproduce: the length of a known route, or a hub that must join its networks. `validate_network` checks a network file against a list of cases.

The pipeline lists three cases for Denmark:

In [10]:
cases = load_validation_cases(osm_region_multimodal.VALIDATION, region="europe/denmark")
for case in cases:
    print(f"{case['id']:<34}{case['kind']:<15}{case['source'][:60]}")

result = validate_network(file, cases)
print()
print(result.summary())

aarhus_to_copenhagen_intermodal   route_lengths  Molslinjen / Geofabrik
arhus_port                        hubs           port: NGA World Port Index (Pub. 150, UpdatedPub150.csv SHA-
aalborg_airport                   hubs           airport: OurAirports (commit 1cdbf9cb), table airports.csv o

PASSED: 3/3 cases OK (failed: 0)


The same check runs at the end of a build with `build_geograph(..., run_validation_cases=True)`, or with `--validate` on the command line. The network then enters the cache only if it passes its cases.

A region of your own has no case in scgraph. Write a few cases for it from routes that you know: the docstring of `scgraph.pipelines.steps.step4_validation_cases` gives the kinds of cases and their rules.

## 6. The streets of a neighbourhood, with turn restrictions

The pipeline keeps the main roads of a region. `load_from_osm` reads the ways that you choose in any OpenStreetMap file. This section loads every street of a part of Paris, between the Place de la Bastille and the Place des Vosges.

In a city, a route must also respect the **turn restrictions**. A turn restriction forbids a turn at a junction, for example a left turn. OpenStreetMap stores it as a **relation**: a record that names the street of arrival, the junction and the street of exit. `load_from_osm` reads these relations.

The next cell downloads the streets of the area and their turn restrictions from the Overpass API of OpenStreetMap. It writes them to one file.

In [11]:
import requests

from scgraph.helpers.osm import get_way_direction

AREA = (48.853, 2.362, 48.862, 2.372)  # south, west, north, east
# The `highway` values of the streets that a car may take
STREETS = {
    "motorway", "trunk", "primary", "secondary", "tertiary", "unclassified", "residential", "living_street",
    "motorway_link", "trunk_link", "primary_link", "secondary_link", "tertiary_link",
}  # fmt: skip

query = f"""[out:xml][timeout:60];
way["highway"~"^({"|".join(sorted(STREETS))})$"]{AREA}->.streets;
rel["type"="restriction"](bw.streets)->.restrictions;
(.streets; .restrictions;);
(._; node(w););
out;"""
# The server refuses a request that does not name its client
answer = requests.post(
    "https://overpass-api.de/api/interpreter",
    data={"data": query},
    headers={"User-Agent": "scgraph-example-notebook"},
    timeout=120,
)
answer.raise_for_status()
with open("streets.osm", "wb") as file:
    file.write(answer.content)
print(f"streets.osm: {len(answer.content) / 1e6:.1f} MB")

streets.osm: 0.3 MB


`select` chooses the ways of the file. It receives the tags of a way, and returns its direction, or None to leave the way out. `get_way_direction` reads the direction of a street from its tags: one way or both ways. `max_km_between_nodes=0.2` cuts a long street, so that two nodes are about 200 m apart.

The cell loads the file twice. `load_turn_restrictions=False` gives the same streets, where a route may make any turn.

In [12]:
def select_streets(tags):
    """The direction of a street, and None for any other way."""
    return get_way_direction(tags) if tags.get("highway") in STREETS else None


streets = GeoGraph.load_from_osm("streets.osm", select=select_streets, max_km_between_nodes=0.2)
streets_any_turn = GeoGraph.load_from_osm(
    "streets.osm", select=select_streets, max_km_between_nodes=0.2, load_turn_restrictions=False
)
print(f"Any turn:                   {len(streets_any_turn.nodes):,} nodes")
print(f"With the turn restrictions: {len(streets.nodes):,} nodes")

Any turn:                   132 nodes
With the turn restrictions: 134 nodes


The network with the turn restrictions has a few more nodes. A junction with a turn restriction has more than one node, at the same point and with the same OpenStreetMap id. The file `scgraph/turn_restrictions.py` explains why, with diagrams. A user of the network sees the effect only: a route makes no forbidden turn.

Here is one such turn. The Rue des Tournelles is a one-way street that ends at the Boulevard Beaumarchais. A driver may not turn left there.

Compare a route on the two networks. It starts in the Rue des Tournelles. It ends on the Boulevard Beaumarchais, to the north of the junction: a left turn away.

In [13]:
TOURNELLES = {"latitude": 48.85771, "longitude": 2.36736}  # in the Rue des Tournelles
BEAUMARCHAIS = {"latitude": 48.85878, "longitude": 2.36778}  # on the Boulevard Beaumarchais

street_routes = {}
for title, geograph in [("Any turn", streets_any_turn), ("With the turn restrictions", streets)]:
    street_routes[title] = geograph.get_shortest_path(origin_node=TOURNELLES, destination_node=BEAUMARCHAIS)
    print(f"{title}: {street_routes[title]['length'] * 1000:.0f} m")

Any turn: 163 m
With the turn restrictions: 307 m


In [14]:
STREET_COLORS = {"Any turn": "#0b0b0b", "With the turn restrictions": "#eb6834"}

street_map = make_map(48.8568, 2.3655, 48.8595, 2.3695, height=520)
for title, route in street_routes.items():
    layer = folium.FeatureGroup(name=f"{title}: {route['length'] * 1000:.0f} m")
    folium.PolyLine(
        route["coordinate_path"],
        color=STREET_COLORS[title],
        weight=4,
        tooltip=f"{title}: {route['length'] * 1000:.0f} m",
    ).add_to(layer)
    layer.add_to(street_map)
for name, place in [("Rue des Tournelles", TOURNELLES), ("Boulevard Beaumarchais", BEAUMARCHAIS)]:
    folium.CircleMarker(
        [place["latitude"], place["longitude"]],
        radius=4,
        color="#0b0b0b",
        fill_opacity=1,
        tooltip=folium.Tooltip(name, permanent=True, direction="top"),
    ).add_to(street_map)
folium.LayerControl(collapsed=False).add_to(street_map)
street_map

Where any turn is allowed, the route turns left at the end of the Rue des Tournelles. With the turn restrictions, it turns right, as a driver must. It turns back at the next junction, and passes the end of the Rue des Tournelles again, on the boulevard. In scgraph, a route may turn back at any node, unless a turn restriction forbids it there.

A search of scgraph runs on this network without a change. `distance_matrix` and `save_as_scgraph` work on it as on any other network.

`load_from_osm` reads the turn restrictions that hold for cars at all hours. It does not read a restriction that holds for some hours only, or for another kind of vehicle. The guide [Networks](https://github.com/connor-makowski/scgraph/blob/main/guides/networks.md), section *Turn Restrictions*, gives the details.

## What to remember

- `GeoGraph.build_geograph("osm_region_multimodal", region=...)` builds the multimodal network of a Geofabrik extract, with the data of the day. It needs `pip install scgraph[osm]`.
- The build keeps the network in the cache. `scgraph_file` gives the path of its file, and `get_scgraph_info` reads its sources and the credit to cite.
- The extract changes every day. Record the name of the file, which is the digest of its content, when a result must be found again.
- `node_modes=[...]` loads some node modes of the network.
- `validate_network` checks the network against known routes.
- `GeoGraph.load_from_osm(file, select=...)` loads the ways that you choose in an OpenStreetMap file. Its routes respect the turn restrictions of the file.

The guide [From open data to a route](https://github.com/connor-makowski/scgraph/blob/main/scgraph/pipelines/README.md), Part 1, follows this build step by step, with its numbers.